# Start

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import glob


from torchvision.transforms import AutoAugment, AutoAugmentPolicy

In [2]:
try:
    # Use the directory of the current script as the project base
    # This is standard when running a .py file
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # Fallback for environments like Jupyter/interactive consoles
    # Assumes the script is being run from the project root
    BASE_DIR = os.getcwd()

# --- 1. CONFIGURAZIONE ---
def find_paths():

    # The 'DATA' folder is the container for everything
    base_data = os.path.join(BASE_DIR, 'DATA')

    # Data for training (inside DATA/dataset)
    base_train = os.path.join(base_data, 'dataset')

    # Data for testing (inside DATA/test_data)
    test_root = os.path.join(base_data, 'test_data')
            
    if test_root is None:
        raise ValueError("Impossibile trovare la cartella test_data/images")

    return {
        'train_img': os.path.join(base_train, 'images'),
        'train_mask': os.path.join(base_train, 'masks'),
        'labels': os.path.join(base_train, 'train_labels.csv'),
        'test_img': os.path.join(test_root, 'test_img'),
        'test_mask': os.path.join(test_root, 'test_mask')
    }

PATHS = find_paths()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Iperparametri Ottimizzati
IMG_SIZE = 256
BATCH_SIZE = 32
N_FOLDS = 5
EPOCHS = 35
N_PATCHES = 4
N_PATCHES_PER_ROI = 20
PATCH_PRECOMPUTED_DIR = os.path.join(BASE_DIR, 'precomputed_patches')
PRECOMPUTED_DIR = os.path.join(os.getcwd(), 'precomputed_rois')

print(f"✅ Project Base Directory: {BASE_DIR}")
print(f"💾 Precomputed ROI Directory: {PRECOMPUTED_DIR}")
print(f"Device type: {DEVICE}" )

✅ Project Base Directory: D:\YANA\CHALLENGE2
💾 Precomputed ROI Directory: D:\YANA\CHALLENGE2\precomputed_rois
Device type: cuda


In [3]:
# Global Default Hyperparameters (Used if a specific classifier doesn't override them)
DEFAULT_HPARAMS = {
    'LR': 1e-4, 
    'WEIGHT_DECAY': 3e-2, 
    'MIXUP_ALPHA': 0.5
}

# --- CLASSIFIER CONFIGURATIONS ---
# Define the three distinct tasks and their initial hyperparameters
CLASSIFIER_CONFIGS = {
    # C1: Triage (Aggressive vs. Luminal Group)
    'C1_TRIAGE': {
        'key': 'C1_TRIAGE',
        'name': 'C1_TRIAGE: Luminal vs. Aggressive',
        'group_0': ['Luminal A', 'Luminal B'],        # Class 0
        'group_1': ['HER2(+)', 'Triple negative'],     # Class 1
        'label_col': 'C1_label',
        'class_names': ['Luminal Group', 'Aggressive Group'],
        'hparams':{
            'LR': 5e-5,          # Lower LR for better convergence on subtle features
            'WEIGHT_DECAY': 3e-2,
            'MIXUP_ALPHA': 0.6   # Increased MixUp for better generalization
        }
    },
    
    # C2: Luminal Subtyping (Requires data where label is Luminal A or B)
    'C2_LUMINAL_SUBTYPE': {
        'key': 'C2_LUMINAL_SUBTYPE',
        'name': 'C2_LUMINAL_SUBTYPE: Luminal A vs. Luminal B',
        'group_0': ['Luminal A'],                      # Class 0
        'group_1': ['Luminal B'],                      # Class 1
        'label_col': 'C2_label',
        'class_names': ['Luminal A', 'Luminal B'],
        'hparams': {
            'LR': 5e-5,          # Very low LR is usually needed for this subtle task
            'WEIGHT_DECAY': 5e-2,    # Increased regularization for the subtle task
            'MIXUP_ALPHA': 0.8       # Max MixUp to fight overfitting on a small subset
        }
    },
    
    # C3: Aggressive Subtyping (Requires data where label is HER2 or TN)
    'C3_AGGRESSIVE_SUBTYPE': {
        'key': 'C3_AGGRESSIVE_SUBTYPE',
        'name': 'C3_AGGRESSIVE_SUBTYPE: HER2(+) vs. Triple Negative',
        'group_0': ['HER2(+)'],                        # Class 0
        'group_1': ['Triple negative'],                # Class 1
        'label_col': 'C3_label',
        'class_names': ['HER2(+)', 'Triple negative'],
        'hparams': {
            'LR': 1e-4,          # Can use a slightly higher LR since the task is clearer
            'WEIGHT_DECAY': 1e-2,    # Less regularization needed
            'MIXUP_ALPHA': 0.4       # Standard MixUp
        }
    }
}

In [4]:
import random
# --- 2. DATASET & PREPROCESSING ---
import torchvision.transforms.functional as TF
from PIL import Image

class ReinhardNormalizer:
    """Normalizzazione Colore per standardizzare i vetrini"""
    def __init__(self):
        self.target_mean = np.array([148.60, 169.30, 105.47])
        self.target_std = np.array([41.56, 11.23, 14.39])

    def __call__(self, img_rgb):
        try:
            img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB).astype("float32")
            img_mean = np.mean(img_lab, axis=(0,1))
            img_std = np.std(img_lab, axis=(0,1)) + 1e-5
            img_lab = (img_lab - img_mean) / img_std
            img_lab = img_lab * self.target_std + self.target_mean
            img_lab = np.clip(img_lab, 0, 255).astype("uint8")
            return cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
        except:
            return img_rgb

reinhard = ReinhardNormalizer()

def get_bbox_from_mask(mask):
    if np.sum(mask) == 0: return None
    rows, cols = np.where(mask > 0)
    return np.min(rows), np.max(rows), np.min(cols), np.max(cols)

def generate_and_save_patches(df, precomputed_roi_dir, output_patch_dir, n_patches, patch_size):
    print("--- 🧠 Starting Patch Generation & Saving ---")

    os.makedirs(output_patch_dir, exist_ok=True)

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Generating Patches"):
        img_name = row['sample_index'] # e.g., 'img_0000.png'
        mask_name = img_name.replace("img", "mask") # e.g., 'mask_0000.png'

        roi_img_path = os.path.join(precomputed_roi_dir, img_name)
        mask_dir = PATHS['train_mask'] # Use the correct mask path for sampling
        mask_path = os.path.join(mask_dir, mask_name)

        try:
            # 1. Load ROI and Mask
            image = Image.open(roi_img_path).convert("RGB")

            # Use the core logic from extract_and_sample_patches to find *valid* patches
            # We will use a dedicated function for sampling coordinates only

            # --- SIMPLIFIED COORDINATE SAMPLING ---
            mask_array = np.array(Image.open(mask_path).convert("L"))
            tissue_coords = np.argwhere(mask_array > 0)

            # Define bounds
            width, height = image.size
            max_x = width - patch_size
            max_y = height - patch_size

            # 2. Sample Coordinates
            if len(tissue_coords) == 0:
                # Skip samples with no tissue
                continue

            for p_idx in range(n_patches):
                # Randomly pick a tissue pixel as the center
                center_y, center_x = random.choice(tissue_coords)

                # Calculate the top-left corner, ensuring it's within bounds
                offset_y = random.randint(-patch_size // 2, patch_size // 2)
                offset_x = random.randint(-patch_size // 2, patch_size // 2)

                start_y = np.clip(center_y - patch_size // 2 + offset_y, 0, max_y)
                start_x = np.clip(center_x - patch_size // 2 + offset_x, 0, max_x)

                # 3. Extract, Save
                patch = TF.crop(image, start_y, start_x, patch_size, patch_size)

                # NEW FILE NAMING: img_0000.png_p01.png
                patch_file_name = f"{img_name.split('.')[0]}_p{p_idx:03d}.png"
                patch_output_path = os.path.join(output_patch_dir, patch_file_name)

                patch.save(patch_output_path, "PNG")

        except Exception as e:
            print(f"Skipping ROI {img_name} due to error: {e}")
            continue

    print(f"--- ✅ Patch Generation Complete. Patches saved to: {output_patch_dir} ---")

def extract_and_sample_patches(
    image: Image,
    mask_path: str, # Path to the corresponding mask file
    patch_size: int = 256,
    n_patches: int = 4,
    min_patch_tissue_area: float = 0.01 # Minimum tissue area required in the extracted patch (1%)
):
    """
    Randomly samples N patches GUARANTEED to contain tissue, or samples randomly
    if the tissue is too sparse to find the required amount.
    """
    try:
        mask_img = Image.open(mask_path).convert("L")
    except FileNotFoundError:
        # If mask is missing, assume the whole ROI is valid, but sampling is random (no guidance)
        mask_img = Image.fromarray(np.ones_like(np.array(image)[:,:,0]) * 255)

    mask = np.array(mask_img)
    width, height = image.size

    # 1. Identify all coordinates where tissue exists (mask is non-zero)
    # This gives us a list of (y, x) coordinates of tissue pixels
    tissue_coords = np.argwhere(mask > 0)

    # Define the boundaries for a patch's top-left corner
    max_x = width - patch_size
    max_y = height - patch_size

    sampled_patches = []

    if len(tissue_coords) == 0:
        # 🛑 FALLBACK: If the mask is entirely empty (no tissue anywhere)
        print(f"Warning: Mask {mask_path} is empty. Returning neutral gray patches.")
        neutral_patch_array = np.ones((patch_size, patch_size, 3), dtype=np.uint8) * 128
        return [Image.fromarray(neutral_patch_array)] * n_patches # Return a stack of neutral patches

    # 2. Iterate to sample N patches
    for _ in range(n_patches):
        # We try to find a good starting coordinate for the patch.
        attempts = 0
        found_valid_patch = False

        while attempts < 100 and not found_valid_patch:
            # Randomly pick a tissue pixel as the center of the patch
            center_y, center_x = random.choice(tissue_coords)

            # Calculate the top-left corner (x, y) based on the center
            # Ensure the top-left corner is within the valid bounds [0, max_dimension - patch_size]

            # Random shift from the center to make the sampling varied
            offset_y = random.randint(-patch_size // 2, patch_size // 2)
            offset_x = random.randint(-patch_size // 2, patch_size // 2)

            start_y = np.clip(center_y - patch_size // 2 + offset_y, 0, max_y)
            start_x = np.clip(center_x - patch_size // 2 + offset_x, 0, max_x)

            # Check the tissue density of the proposed patch area
            mask_patch = mask[start_y : start_y + patch_size, start_x : start_x + patch_size]
            tissue_area_ratio = np.count_nonzero(mask_patch) / (patch_size * patch_size)

            if tissue_area_ratio >= min_patch_tissue_area:
                image_patch = TF.crop(image, start_y, start_x, patch_size, patch_size)
                sampled_patches.append(image_patch)
                found_valid_patch = True

            attempts += 1

        # 3. Fallback for the current patch if 100 attempts failed (rare for a tissue-rich ROI)
        if not found_valid_patch:
            # If we couldn't find a tissue-rich patch even after 100 attempts,
            # fall back to a random patch from the whole ROI.
            start_y = random.randint(0, max_y)
            start_x = random.randint(0, max_x)
            image_patch = TF.crop(image, start_y, start_x, patch_size, patch_size)
            sampled_patches.append(image_patch)
            # You might log a warning here if this happens often:
            # print(f"Warning: Failed to find tissue-rich patch for {mask_path}. Using random patch.")

    # This array is guaranteed to have exactly n_patches elements.
    return sampled_patches

# class TissueDataset(Dataset):
#     def __init__(self, dataframe, precomputed_dir, transform=None, is_test=False, target_label_column='C1_label'):
#         self.df = dataframe
#         # This directory now contains the final, preprocessed images
#         self.precomputed_dir = precomputed_dir
#         self.transform = transform
#         self.is_test = is_test
#         self.target_label_column = target_label_column
#
#     def __len__(self): return len(self.df)
#
#     def __getitem__(self, idx):
#         row = self.df.iloc[idx]
#         img_name = row['sample_index']
#         mask_name = img_name.replace("img", "mask")
#         img_path = os.path.join(self.precomputed_dir, img_name)
#         # Use the globally defined PATHS to find the mask directory
#         mask_dir = PATHS['train_mask'] if not self.is_test else PATHS['test_mask']
#         mask_path = os.path.join(mask_dir, mask_name)
#
#         # Load the high-resolution ROI (e.g., 768x768 or 1024x1024)
#         try:
#             # Load the high-resolution ROI (assumed to be in precomputed_dir)
#             image = Image.open(img_path).convert("RGB")
#         except Exception:
#             # Fallback for missing image file (returns a stack of black tensors)
#             black_patch = torch.zeros(3, IMG_SIZE, IMG_SIZE)
#             black_stack = torch.stack([black_patch] * 9) # 9 black patches
#             if self.is_test: return black_stack, row['sample_index']
#             else: return black_stack, torch.tensor(row[self.target_label_column], dtype=torch.long)
#
#         # 1. EXTRACT PATCHES (e.g., 9 patches of 256x256)
#         patches = extract_and_sample_patches( # 🛑 CHANGED FUNCTION
#             image,
#             mask_path=mask_path,
#             patch_size=IMG_SIZE,
#             n_patches=N_PATCHES,
#             min_patch_tissue_area=0.01 # Using 1% for robustness
#         )
#
#         # 2. APPLY TRANSFORMS TO EACH PATCH
#         transformed_patches = []
#         for patch in patches:
#             # We must use the image-level transforms here (ToTensor, Normalize, Augmentations)
#             if self.transform:
#                 transformed_patches.append(self.transform(patch))
#
#         # 3. STACK PATCHES: Combine the list of (C, H, W) tensors into one (N_PATCHES, C, H, W) tensor
#         image_tensor = torch.stack(transformed_patches)
#
#         if self.is_test:
#             return image_tensor, row['sample_index']
#         else:
#             label = row[self.target_label_column]
#             return image_tensor, torch.tensor(label, dtype=torch.long)

class SimplePatchDataset(Dataset):
    def __init__(self, patch_files, df_labels, transform=None, is_test=False, target_label_column='C1_label'):
        self.patch_files = patch_files
        self.transform = transform
        self.is_test = is_test
        self.target_label_column = target_label_column

        # Map original image name (img_0000.png) to its label row
        # This requires reverse-engineering the label from the patch filename
        self.label_map = df_labels.set_index('sample_index')[target_label_column].to_dict()

    def __len__(self):
        return len(self.patch_files)

    def __getitem__(self, idx):
        patch_path = self.patch_files[idx]

        # Extract the original ROI name from the patch name (e.g., 'img_0000_p001.png' -> 'img_0000.png')
        base_img_name = os.path.basename(patch_path).split('_p')[0] + '.png'

        # Load image (now simple I/O)
        image = Image.open(patch_path).convert("RGB")

        if self.transform:
            image_tensor = self.transform(image)
        else:
            image_tensor = transforms.ToTensor()(image)

        if self.is_test:
            return image_tensor, os.path.basename(patch_path) # Return patch name for tracking
        else:
            label = self.label_map.get(base_img_name)
            return image_tensor, torch.tensor(label, dtype=torch.long)


def aggregate_patch_predictions(patch_preds, patch_paths, df_labels, label_column):
    """
    Groups patch predictions by their parent ROI and applies a majority vote.
    Returns aggregated ROI predictions and the true ROI targets.
    """
    results_df = pd.DataFrame({
        'patch_pred': patch_preds,
        'patch_path': patch_paths
    })

    # 1. Map patch path back to the original ROI name
    results_df['sample_index'] = results_df['patch_path'].apply(
        lambda x: os.path.basename(x).split('_p')[0] + '.png'
    )

    # 2. Get true ROI target by merging with the label DataFrame
    # Note: df_labels is df_task, already filtered for relevant samples.
    targets_df = df_labels[['sample_index', label_column]].drop_duplicates()
    results_df = results_df.merge(targets_df, on='sample_index', how='left')

    # 3. Aggregation (Majority Voting)
    roi_results = results_df.groupby('sample_index').agg(
        # The mean prediction (0 or 1) gives us the voting result. Mean > 0.5 means majority voted 1.
        roi_pred=('patch_pred', 'mean'),
        roi_target=(label_column, 'first')
    ).reset_index()

    # Convert mean vote (0.0 to 1.0) into final binary prediction (0 or 1)
    roi_results['final_pred'] = (roi_results['roi_pred'] >= 0.5).astype(int)

    return roi_results['final_pred'].tolist(), roi_results['roi_target'].tolist()

In [5]:
# --- 3. TRANSFORMS (MODIFIED) ---
def get_transforms():
    # Example transforms for single patch input
    train_tfms = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(90),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    val_tfms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return train_tfms, val_tfms

def mixup_data(x, y, alpha):
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    index = torch.randperm(x.size(0)).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index, :]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


In [6]:
# --- 4. MODELLO (MODIFIED FOR PATCH AGGREGATION) ---
# class PatchClassifier(nn.Module):
#     def __init__(self, base_model, num_classes):
#         super().__init__()
#         # Use everything up to the classifier (the feature extractor)
#         self.features = base_model.features
#         in_features = base_model.classifier[1].in_features
#
#         # Define the patch-level classification head
#         self.head = nn.Sequential(
#             nn.Dropout(0.2),
#             nn.Linear(in_features, 256),
#             nn.ReLU(),
#             nn.Linear(256, 1) # Single output logit for binary classification
#         )
#
#     def forward(self, x):
#         # Input shape: (Batch_Size, N_Patches, C, H, W)
#         B, N, C, H, W = x.shape
#         x = x.view(B * N, C, H, W) # Flatten patches into batch dimension
#
#         # 1. Feature Extraction (Base Model)
#         features = self.features(x)
#         features = features.mean([2, 3]) # Global Average Pooling
#
#         # 2. Patch-Level Prediction
#         logits = self.head(features)
#
#         # 3. AGGREGATION (Mean Pooling)
#         # Reshape back to (B, N_Patches, 1) and average across patches
#         logits = logits.view(B, N, 1)
#         roi_prediction = logits.mean(dim=1) # Output shape: (B, 1)
#
#         return roi_prediction

def get_model(num_classes):
    # This must be run on the CPU/GPU device
    base_model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

    # Replace the classification head for binary output (1 logit)
    in_features = base_model.classifier[1].in_features
    base_model.classifier = nn.Sequential(
        nn.Dropout(0.2),
        nn.Linear(in_features, 1)
    )
    return base_model.to(DEVICE)

In [7]:
# --- 5 PRECOMPUTATION FUNCTION ---

def preprocess_and_save_images(df, paths, output_dir, img_size):
    """
    Applies ROI extraction and Reinhard normalization once, saving the result 
    to a new output_dir so the DataLoader only handles fast reads and PyTorch transforms.
    """
    print("Starting precomputation and saving of processed ROIs...")
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize the Reinhard Normalizer
    reinhard = ReinhardNormalizer()
    
    # We will track which images were successfully processed
    processed_indices = []
    print(f"DEBUG: Target Resize Dimension set to: {img_size}")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing Images"):
        img_name = row['sample_index']
        mask_name = img_name.replace("img", "mask")
        
        img_path = os.path.join(paths['train_img'], img_name)
        mask_path = os.path.join(paths['train_mask'], mask_name)
        output_path = os.path.join(output_dir, img_name)
        
        try:
            # 1. Read and Convert
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            # 2. ROI Extraction (using mask)
            if mask is not None:
                bbox = get_bbox_from_mask(mask)
                if bbox:
                    y_min, y_max, x_min, x_max = bbox
                    image = image[y_min:y_max+1, x_min:x_max+1]
            
            # 3. Color Normalization
            image = reinhard(image)
            
            # 4. Resize and Save (as PNG for better quality than JPEG)
            # Use PIL for robust and standard resizing for saving
            image_pil = Image.fromarray(image)

            # 🛑 DEBUG PRINT 2: Check image size before resizing (should be the cropped ROI size)
            if idx == 0: # Only print for the first image for brevity
                print(f"DEBUG: Image {img_name} size before resize: {image_pil.size}")

            # Resize the cropped ROI to the target size before saving
            image_resized = image_pil.resize((img_size, img_size), Image.Resampling.LANCZOS)

            # 🛑 DEBUG PRINT 3: Check image size after resizing (should be 768x768)
            if idx == 0: # Only print for the first image for brevity
                print(f"DEBUG: Image {img_name} size AFTER resize: {image_resized.size}")

            # Resize the cropped ROI to the target size before saving
            image_resized = image_pil.resize((img_size, img_size), Image.Resampling.LANCZOS)
            
            image_resized.save(output_path, "PNG")
            processed_indices.append(idx)
            
        except Exception as e:
            # Handle cases where image/mask read fails
            print(f"Skipping image {img_name} due to error: {e}")
            continue

    # Return the filtered dataframe containing only successfully processed images
    return df.iloc[processed_indices].reset_index(drop=True)


def setup_data_and_preprocess():
    """
    Reads labels, performs binary grouping, runs the heavy CPU precomputation
    (ROI extraction and Reinhard normalization), and returns the filtered DataFrame.
    """
    print("--- 🧠 Data Setup & Preprocessing Started ---")
    
    # 1. Load Data and Filter by Existence (Unchanged)
    df = pd.read_csv(PATHS['labels'])
    valid_mask = df['sample_index'].apply(lambda x: os.path.exists(os.path.join(PATHS['train_img'], x)))
    df = df[valid_mask].reset_index(drop=True)

    # 2. Multi-Task Labeling: Create 3 Binary Labels (NEW LOGIC)
    for key, config in CLASSIFIER_CONFIGS.items():
        # Combine all classes used in this task
        all_classes_in_task = config['group_0'] + config['group_1']
        
        # Filter out rows that are not relevant to this task for C2 and C3
        # For C1, all rows are relevant. For C2/C3, irrelevant rows will be marked -1.
        
        def get_classifier_label(original_label):
            if original_label in config['group_0']:
                return 0
            elif original_label in config['group_1']:
                return 1
            else:
                return -1 # Mark rows irrelevant to this task
                
        df[config['label_col']] = df['label'].apply(get_classifier_label)

    # 3. Precompute ROIs, Resize, and Save (Unchanged)
    # The precomputation step doesn't care about the labels, only the images.
# New Configuration Variable
    ROI_TARGET_SIZE = 768 # e.g., 3 patches (256) x 3 patches (256) = 768
    IMG_SIZE = 256         # This remains the patch size

    df_final = preprocess_and_save_images(df, PATHS, PRECOMPUTED_DIR, ROI_TARGET_SIZE)
    # Filter out any rows that failed precomputation
    # Note: df_final now contains all 3 label columns (C1_label, C2_label, C3_label)
    
    print(f"--- ✅ Precomputation Complete. Final images available: {len(df_final)} ---")
    return df_final

In [8]:
# RUN THIS ONCE BEFORE STARTING K-FOLD
# This creates a dictionary mapping parent ROI to its list of patch paths

def create_patch_index(patch_dir):
    patch_index = {}
    all_patch_files = glob.glob(os.path.join(patch_dir, '*.png'))

    for f in all_patch_files:
        # Get parent ROI name (e.g., 'img_0000.png')
        roi_name = os.path.basename(f).split('_p')[0] + '.png'

        if roi_name not in patch_index:
            patch_index[roi_name] = []
        patch_index[roi_name].append(f)

    return patch_index

# Global Variable:

In [26]:
import time
# --- 6. TRAINING LOOP ---
import glob

def train_fold(fold, train_idx, val_idx, df_task, class_names, precomputed_dir, config):
    classifier_key = config['key']
    label_col = config['label_col']
    print(f"\n=== FOLD {fold+1}/{N_FOLDS} ===")
    start_time = time.time() # Start timer

    train_df = df_task.iloc[train_idx].reset_index(drop=True)
    val_df = df_task.iloc[val_idx].reset_index(drop=True)
    
    # Filter dataframes to only include relevant samples for this task (label != -1)
    # train_df = train_df[train_df[config['label_col']] != -1].reset_index(drop=True)
    # val_df = val_df[val_df[config['label_col']] != -1].reset_index(drop=True)
    
    # Check if the filtered dataframes still have samples
    if len(train_df) == 0 or len(val_df) == 0:
        print("Skipping fold: Insufficient data for this classifier's task.")
        return None, [], [] # Return placeholders if no data remains for this task

    # --- NEW PATCH FILE LIST GENERATION ---
    # all_patch_files = glob.glob(os.path.join(PATCH_PRECOMPUTED_DIR, '*.png'))
    #
    # # Get the list of original ROI names for the current fold
    # train_roi_names = set(train_df['sample_index'].tolist())
    # val_roi_names = set(val_df['sample_index'].tolist())
    #
    # # Filter patch files based on which ROI they belong to
    # train_patch_files = [
    #     f for f in all_patch_files
    #     if os.path.basename(f).split('_p')[0] + '.png' in train_roi_names
    # ]
    # val_patch_files = [
    #     f for f in all_patch_files
    #     if os.path.basename(f).split('_p')[0] + '.png' in val_roi_names
    # ]

    train_roi_names = set(train_df['sample_index'].tolist())
    val_roi_names = set(val_df['sample_index'].tolist())

    train_patch_files = []
    for roi_name in train_roi_names:
        if roi_name in PATCH_INDEX:
            train_patch_files.extend(PATCH_INDEX[roi_name])

    val_patch_files = []
    for roi_name in val_roi_names:
        if roi_name in PATCH_INDEX:
            val_patch_files.extend(PATCH_INDEX[roi_name])

    print(f"DEBUG (P1): Data Preparation Complete. Train Patches: {len(train_patch_files)} | Time: {time.time() - start_time:.2f}s")

    print(f"Training on {len(train_patch_files)} patches | Validating on {len(val_patch_files)} patches")

    if len(train_patch_files) == 0 or len(val_patch_files) == 0:
        print("Skipping fold: Insufficient patch data for this task.")
        return None, [], [], []

    # Class Balancing
    roi_label_map = train_df.set_index('sample_index')[config['label_col']].to_dict()

    train_patch_labels = [
        roi_label_map[os.path.basename(f).split('_p')[0] + '.png']
        for f in train_patch_files
    ]

    class_counts = pd.Series(train_patch_labels).value_counts().sort_index().values
    class_weights = 1. / class_counts
    sample_weights = [class_weights[l] for l in train_patch_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    # 3. Data Loaders (Use the new SimplePatchDataset)
    train_loader = DataLoader(SimplePatchDataset(train_patch_files, train_df, get_transforms()[0], target_label_column=label_col),
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=0)
    val_loader = DataLoader(SimplePatchDataset(val_patch_files, val_df, get_transforms()[1], target_label_column=label_col),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # 3. HParams (Use the config dictionary)
    HPARAMS = config['hparams']
    
    model = get_model(num_classes = 1)
    optimizer = optim.AdamW(model.parameters(), lr=HPARAMS['LR'], weight_decay = HPARAMS['WEIGHT_DECAY'])
    criterion = nn.BCEWithLogitsLoss()
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    
    history = {'t_loss': [], 'v_loss': [], 't_f1': [], 'v_f1': []}
    best_f1 = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        t_loss = 0
        t_preds, t_targets = [], []

        print(f"DEBUG (P3): Epoch {epoch+1} starting loop. Waiting for first batch...")
        batch_start_time = time.time()

        for imgs, lbls in tqdm(train_loader, desc=f"Ep {epoch+1}", leave=False):
            # if batch_idx == 0:
            #     print(f"DEBUG (P4): Batch 0 loaded by DataLoader! CPU Time spent waiting: {time.time() - batch_start_time:.2f}s")
            #     # If this time is huge (e.g., > 60s), the bottleneck is inside DataLoader workers (SimplePatchDataset)

            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)

            # if batch_idx == 0:
            #     print(f"DEBUG (P5): Batch 0 transferred to GPU. Time for transfer+prep: {time.time() - batch_start_time:.2f}s")
            #     # If the jump from P4 to P5 is huge, the bottleneck is the .to(DEVICE) transfer or pre-mixup prep.

            optimizer.zero_grad()
            
            # --- MIXUP BLOCK ---
            lbls_float = lbls.float().unsqueeze(1)
            imgs, targets_a, targets_b, lam = mixup_data(imgs, lbls_float, HPARAMS['MIXUP_ALPHA'])
            out = model(imgs)
            loss = mixup_criterion(criterion, out, targets_a, targets_b, lam)
            
            loss.backward()
            optimizer.step()

            # if batch_idx == 0:
            #     print(f"DEBUG (P6): Batch 0 completed. Time: {time.time() - batch_start_time:.2f}s")

            t_loss += loss.item()
            
            # Per le metriche usiamo l'argmax approssimato (non perfetto col mixup ma indicativo)
            t_preds.extend((torch.sigmoid(out) >= 0.5).long().reshape(-1).cpu().numpy())
            t_targets.extend(targets_a.round().long().reshape(-1).cpu().numpy()) # targets_a is float

        scheduler.step()
        
        # Validation (No Mixup)
        model.eval()
        v_loss = 0
        v_patch_preds, v_patch_paths = [], []

        # 🛑 Temporary Fix: Assume the val_loader index matches val_patch_files order.
        # This is often true when shuffle=False, but it's risky!
        val_patch_paths_global = val_patch_files

        current_patch_idx = 0
        with torch.no_grad():
            # 🛑 CRITICAL FIX: UNPACK ONLY 2 VALUES 🛑
            for imgs, lbls in val_loader:

                # --- Temporary Fix: Manually retrieve the patch paths for the batch ---
                batch_size = imgs.size(0)

                # Extract the correct paths from the global list based on the batch size
                current_batch_paths = val_patch_paths_global[
                    current_patch_idx : current_patch_idx + batch_size
                ]
                current_patch_idx += batch_size
                # ---------------------------------------------------------------------

                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE) # <-- Clean transfer
                lbls_float = lbls.float().unsqueeze(1)

                out = model(imgs)
                v_loss += criterion(out, lbls_float).item()

                v_patch_preds.extend((torch.sigmoid(out) >= 0.5).long().reshape(-1).cpu().numpy())

                # Store the manually retrieved paths for aggregation
                v_patch_paths.extend(current_batch_paths)
        # with torch.no_grad():
        #     for imgs, lbls in val_loader: # patch_paths is returned by SimplePatchDataset
        #         # imgs = imgs.to(DEVICE)
        #         #
        #         # # We need the true target labels for the V_Loss calculation
        #         # # This is a bit inefficient, but required since DataLoader gives paths, not targets
        #         # batch_targets = torch.tensor([roi_label_map[os.path.basename(p).split('_p')[0] + '.png'] for p in patch_paths], dtype=torch.long).to(DEVICE)
        #
        #         imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE) # <-- Clean transfer
        #         lbls_float = lbls.float().unsqueeze(1)
        #
        #         out = model(imgs)
        #         v_loss += criterion(out, lbls_float).item()
        #
        #         v_patch_preds.extend((torch.sigmoid(out) >= 0.5).long().reshape(-1).cpu().numpy())

        v_roi_preds, v_roi_targets = aggregate_patch_predictions(v_patch_preds, v_patch_paths, df_task, label_col)

        # 4. Metric Calculation
        t_loss /= len(train_loader)
        v_loss /= len(val_loader)
        t_f1 = f1_score(t_targets, t_preds, average='macro')
        # V_F1 must be calculated on the aggregated ROI predictions
        v_f1 = f1_score(v_roi_targets, v_roi_preds, average='macro')

        history['t_loss'].append(t_loss); history['v_loss'].append(v_loss)
        history['t_f1'].append(t_f1); history['v_f1'].append(v_f1)

        print(f"Ep {epoch+1}: T_Loss {t_loss:.3f} | T_F1 {t_f1:.3f} | V_Loss {v_loss:.3f} | V_F1 (ROI) {v_f1:.3f}")

        if v_f1 > best_f1:
            best_f1 = v_f1
            torch.save(model.state_dict(), f"{classifier_key}_fold_{fold}_best.pth")

    print(f"Best Val F1 (ROI): {best_f1:.4f}")

    # Return the aggregated ROI predictions and the true ROI targets for the final K-Fold aggregation
    return history, v_roi_preds, v_roi_targets

In [10]:
# --- 7. EXECUTION ---

def train_classifier(classifier_key, df_final, precomputed_dir):
    
    config = CLASSIFIER_CONFIGS[classifier_key]
    print(f"\n=======================================================")
    print(f"       STARTING TRAINING FOR: {config['name']}       ")
    print(f"       HPARAMS: {config['hparams']}  ")
    print(f"=======================================================")

    # Create a simple mapping for plotting labels
    class_names = config['class_names']
    
    # Dummy LabelEncoder for plotting (since we only output 1 unit)
    class DummyLabelEncoder:
        def __init__(self, names):
            self.classes_ = names
        def inverse_transform(self, predictions):
            return [self.classes_[p] for p in predictions]
    le = DummyLabelEncoder(class_names)
    
    df = df_final
    df_task = df[df[config['label_col']] != -1].reset_index(drop=True)

    if len(df_task) == 0:
        print("No samples available for this classifier after filtering.")
        return None, le

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    histories, all_preds, all_targets = [], [], [] # all_preds/all_targets are now ROI-level

    # Use the specific label column for stratification
    for fold, (t_idx, v_idx) in enumerate(skf.split(df_task, df_task[config['label_col']])):
        # train_fold MUST NOW return patch_preds and patch_paths from the validation set
        h, p, t = train_fold(fold, t_idx, v_idx, df_task, class_names, precomputed_dir, config)
        if h is not None:
            histories.append(h)
            all_preds.extend(p)
            all_targets.extend(t)
            
    # --- PLOTTING (Use the result of one run to demonstrate metrics) ---
    if not histories:
        print("Training failed for all folds due to lack of data.")
        return None, le
        
    # Plots... (remains the same as your original plotting block, just use histories/targets/preds)
    fig, axes = plt.subplots(N_FOLDS, 2, figsize=(12, 4*N_FOLDS))
    for i, h in enumerate(histories):
        axes[i][0].plot(h['t_loss'], label='Train')
        axes[i][0].plot(h['v_loss'], label='Val')
        axes[i][0].legend(); axes[i][0].set_title(f'Fold {i+1} Loss')
        axes[i][1].plot(h['t_f1'], label='Train F1')
        axes[i][1].plot(h['v_f1'], label='Val F1')
        axes[i][1].legend(); axes[i][1].set_title(f'Fold {i+1} F1')
    plt.tight_layout()
    plt.show()
    
    cm = confusion_matrix(all_targets, all_preds)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_)
    plt.show()
    
    return all_preds, le

In [11]:
# 1. Initial Setup (Call once)
DF_FINAL = setup_data_and_preprocess()

--- 🧠 Data Setup & Preprocessing Started ---
Starting precomputation and saving of processed ROIs...
DEBUG: Target Resize Dimension set to: 768


Preprocessing Images:   0%|          | 1/581 [00:00<01:13,  7.87it/s]

DEBUG: Image img_0000.png size before resize: (407, 701)
DEBUG: Image img_0000.png size AFTER resize: (768, 768)


Preprocessing Images: 100%|██████████| 581/581 [00:44<00:00, 12.91it/s]

--- ✅ Precomputation Complete. Final images available: 581 ---


In [12]:
generate_and_save_patches(
    DF_FINAL,
    PRECOMPUTED_DIR,
    PATCH_PRECOMPUTED_DIR,
    N_PATCHES_PER_ROI,
    IMG_SIZE
)

PATCH_INDEX = create_patch_index(PATCH_PRECOMPUTED_DIR)

--- 🧠 Starting Patch Generation & Saving ---


Generating Patches: 100%|██████████| 581/581 [00:51<00:00, 11.26it/s]


--- ✅ Patch Generation Complete. Patches saved to: D:\YANA\CHALLENGE2\precomputed_patches ---


In [27]:
# 2. TRAINING (Called three times, one for each model)
# Train Classifier 1
if __name__ == "__main__":
    preds_c1, le_c1 = train_classifier('C1_TRIAGE', DF_FINAL, PRECOMPUTED_DIR)


       STARTING TRAINING FOR: C1_TRIAGE: Luminal vs. Aggressive       
       HPARAMS: {'LR': 5e-05, 'WEIGHT_DECAY': 0.03, 'MIXUP_ALPHA': 0.6}  

=== FOLD 1/5 ===
DEBUG (P1): Data Preparation Complete. Train Patches: 9280 | Time: 0.00s
Training on 9280 patches | Validating on 2340 patches
DEBUG (P3): Epoch 1 starting loop. Waiting for first batch...


Ep 1: T_Loss 0.677 | T_F1 0.548 | V_Loss 0.694 | V_F1 (ROI) 0.566
DEBUG (P3): Epoch 2 starting loop. Waiting for first batch...


Ep 2: T_Loss 0.642 | T_F1 0.580 | V_Loss 0.652 | V_F1 (ROI) 0.654
DEBUG (P3): Epoch 3 starting loop. Waiting for first batch...


Ep 3: T_Loss 0.610 | T_F1 0.597 | V_Loss 0.716 | V_F1 (ROI) 0.595
DEBUG (P3): Epoch 4 starting loop. Waiting for first batch...


Ep 4: T_Loss 0.586 | T_F1 0.629 | V_Loss 0.677 | V_F1 (ROI) 0.606
DEBUG (P3): Epoch 5 starting loop. Waiting for first batch...


...

Ep 35: T_Loss 0.400 | T_F1 0.725 | V_Loss 0.755 | V_F1 (ROI) 0.662
Best Val F1 (ROI): 0.6621

=== FOLD 2/5 ===
DEBUG (P1): Data Preparation Complete. Train Patches: 9300 | Time: 0.00s
Training on 9300 patches | Validating on 2320 patches
DEBUG (P3): Epoch 1 starting loop. Waiting for first batch...


Ep 1: T_Loss 0.674 | T_F1 0.551 | V_Loss 0.668 | V_F1 (ROI) 0.583
DEBUG (P3): Epoch 2 starting loop. Waiting for first batch...


Ep 2: T_Loss 0.640 | T_F1 0.586 | V_Loss 0.714 | V_F1 (ROI) 0.584
DEBUG (P3): Epoch 3 starting loop. Waiting for first batch...


Ep 3: T_Loss 0.603 | T_F1 0.608 | V_Loss 0.814 | V_F1 (ROI) 0.604
DEBUG (P3): Epoch 4 starting loop. Waiting for first batch...


KeyboardInterrupt: 

In [ ]:
# Train Classifier 2 (Luminal Subtypes)
preds_c2, le_c2 = train_classifier('C2_LUMINAL_SUBTYPE', DF_FINAL, PRECOMPUTED_DIR)

In [ ]:
# Train Classifier 3 (Aggressive Subtypes)
preds_c3, le_c3 = train_classifier('C3_AGGRESSIVE_SUBTYPE', DF_FINAL, PRECOMPUTED_DIR)

# Tuna salad

In [ ]:
import optuna


# --- Unified Optuna Objective Function ---
def objective_classifier(trial, classifier_key):
    """
    Defines the search space and runs a single fold of training for the specified classifier.
    Returns the final validation F1 score.
    """

    # --- 1. Define Search Space based on the task (Can be adjusted) ---
    hparams = {
        # Search LR (Logarithmic scale is best for LR)
        'LR': trial.suggest_float('LR', 1e-5, 5e-4, log=True),
        # Search WEIGHT_DECAY (L2 Regularization)
        'WEIGHT_DECAY': trial.suggest_float('WEIGHT_DECAY', 1e-4, 1e-1, log=True),
        # Search MIXUP_ALPHA (Strength of data augmentation)
        'MIXUP_ALPHA': trial.suggest_float('MIXUP_ALPHA', 0.2, 0.9)
    }

    # --- 2. Setup Configuration ---
    config = CLASSIFIER_CONFIGS[classifier_key].copy()
    config['hparams'] = hparams # Override hparams with trial suggestions

    # --- 3. Prepare Data (Filter for the task) ---
    df_task = DF_FINAL[DF_FINAL[config['label_col']] != -1].reset_index(drop=True)

    if len(df_task) == 0:
        print(f"Objective skipped for {classifier_key}: No relevant samples.")
        return 0.0

    # --- 4. Use a single fixed fold for evaluation (Fold 0) ---
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

    # Get the indices for the first fold (Fold 0)
    try:
        t_idx, v_idx = next(skf.split(df_task, df_task[config['label_col']]))
    except StopIteration:
        print(f"Objective skipped for {classifier_key}: StratifiedKFold failed.")
        return 0.0

    # --- 5. Run Training (using the existing train_fold function) ---
    # NOTE: train_fold is expected to correctly use the HPARAMS from the 'config' dict.
    class_names = config['class_names']

    h, _, _ = train_fold(
        fold=0,
        train_idx=t_idx,
        val_idx=v_idx,
        df=df_task,
        le=class_names, # 'le' is used as class_names in train_fold
        precomputed_dir=PRECOMPUTED_DIR,
        config=config
    )

    # --- 6. Return Validation F1 Score ---
    if h is None:
        return 0.0 # Failed trial

    # Optuna maximizes the validation F1 score from the last epoch
    return h['v_f1'][-1]

# --- Optuna Study Runner (Wrapper for running the study) ---
def tune_classifier(classifier_key, n_trials=20):
    """
    Sets up and executes the Optuna study for the specified classifier.
    """
    print(f"\n=======================================================")
    print(f"       STARTING OPTUNA TUNING FOR: {classifier_key}          ")
    print(f"       Running {n_trials} trials (using 1 fold/trial)      ")
    print(f"=======================================================")

    # Create a wrapper function for the objective that fixes the classifier_key
    func = lambda trial: objective_classifier(trial, classifier_key)

    study = optuna.create_study(direction="maximize", study_name=f"tune_{classifier_key}")
    study.optimize(func, n_trials=n_trials)

    print(f"\n--- Optuna Study for {classifier_key} Finished ---")
    print(f"Best Trial F1 Score: {study.best_value:.4f}")

    best_params = study.best_params
    print("Best Hyperparameters Found:")
    for key, value in best_params.items():
        print(f"  {key}: {value:.6f}")

    # Update the global config for the specific classifier
    CLASSIFIER_CONFIGS[classifier_key]['hparams'].update(best_params)

    return best_params

In [ ]:
# --- 1. Tune C1: Triage (Luminal vs. Aggressive Group) ---
best_c1_hparams = tune_classifier('C1_TRIAGE', n_trials=30) 


# --- 4. Final Production Training (Saves 5 models for each classifier) ---

print("\n--- Starting FINAL 5-FOLD PRODUCTION TRAINING for all three classifiers ---")

# C1 Final Training (Saves C1_TRIAGE_fold_X_best.pth)
train_classifier('C1_TRIAGE', DF_FINAL, PRECOMPUTED_DIR)

In [ ]:
# --- 2. Tune C2: Luminal Subtyping (Luminal A vs. Luminal B) ---
# Note: C2 usually needs more trials and stronger regularization (MixUp/WD).
best_c2_hparams = tune_classifier('C2_LUMINAL_SUBTYPE', n_trials=40) 

# C2 Final Training (Saves C2_LUMINAL_SUBTYPE_fold_X_best.pth)
train_classifier('C2_LUMINAL_SUBTYPE', DF_FINAL, PRECOMPUTED_DIR)

In [ ]:
# --- 3. Tune C3: Aggressive Subtyping (HER2(+) vs. Triple Negative) ---
best_c3_hparams = tune_classifier('C3_AGGRESSIVE_SUBTYPE', n_trials=30) 

# C3 Final Training (Saves C3_AGGRESSIVE_SUBTYPE_fold_X_best.pth)
train_classifier('C3_AGGRESSIVE_SUBTYPE', DF_FINAL, PRECOMPUTED_DIR)

In [ ]:
print("\n✅ All 15 models (5 folds x 3 classifiers) saved successfully!")

# SUBMISSION

In [ ]:
# --- ADD THIS TO YOUR GLOBAL CONSTANTS SECTION ---
TEST_PRECOMPUTED_DIR = '/kaggle/working/precomputed_rois_test'

DF_TEST_PROCESSED = None

In [ ]:
# --- New Function: Setup Test Data and Preprocess Images ---
def setup_test_data_and_preprocess():
    """
    Finds test image files, runs the heavy CPU precomputation (ROI extraction and 
    Reinhard normalization), and returns the DataFrame of successfully processed test images.
    """
    print("--- 🧠 Test Data Setup & Preprocessing Started ---")
    
    # 1. List Test Files
    test_files = [f for f in os.listdir(PATHS['test_img']) if f.endswith('.png')]
    # Create a DataFrame required by preprocess_and_save_images
    test_df = pd.DataFrame({'sample_index': test_files})
    
    # 2. Precompute ROIs, Resize, and Save (Reusing the core logic)
    # We must adapt the existing 'preprocess_and_save_images' slightly 
    # to accept dynamic image/mask paths since it was hardcoded for 'train_img'/'train_mask'.
    
    # --- TEMPORARY FIX: Override PATHS for test processing ---
    # We temporarily create a local path structure that preprocess_and_save_images can use.
    local_paths = {
        'train_img': PATHS['test_img'],    # Source folder for images
        'train_mask': PATHS['test_mask']   # Source folder for masks
    }

    # You must update preprocess_and_save_images in your code to accept and use this local_paths dict
    # (assuming it uses the 'train_img'/'train_mask' keys internally).
    
    df_test_final = preprocess_and_save_images(
        df=test_df, 
        paths=local_paths, # Pass the test paths
        output_dir=TEST_PRECOMPUTED_DIR, 
        img_size=IMG_SIZE
    )
    
    print(f"--- ✅ Test Precomputation Complete. Final test images available: {len(df_test_final)} ---")
    return df_test_final

In [ ]:
# --- NEW: Function to Check/Run Test Preprocessing Once ---
def get_processed_test_data():
    """
    Checks if the test data has been processed. If not, runs the heavy preprocessing 
    and stores the resulting DataFrame globally.
    """
    global DF_TEST_PROCESSED
    
    if DF_TEST_PROCESSED is None:
        print("--- ⏳ First call: Running Test Data Preprocessing ---")
        # Call your existing function that does the work (and includes the fix)
        DF_TEST_PROCESSED = setup_test_data_and_preprocess()
        
        if DF_TEST_PROCESSED.empty:
            raise ValueError("Test preprocessing returned an empty DataFrame. Cannot generate submission.")
        print(f"--- ✅ Test Data Stored. Total processed images: {len(DF_TEST_PROCESSED)} ---")
    else:
        print("--- ⏩ Skipping Test Preprocessing (Already done) ---")
        
    return DF_TEST_PROCESSED

In [ ]:
# --- New Function: Prediction Helper (TTA + Ensemble for one task) ---
def predict_with_ensemble_tta(loader, model_prefix):
    """
    Loads 5 models (folds) for a specific task and computes the mean prediction
    using 4x TTA. Returns the raw logits/probabilities (for binary class 1).
    """
    models_list = []
    
    # Load 5 folds for the specific model_prefix (e.g., 'C1_TRIAGE_fold_i_best.pth')
    for i in range(N_FOLDS):
        try:
            m = get_model(num_classes=1) # All classifiers are binary (1 output unit)
            # *** Load the model file saved during training ***
            m.load_state_dict(torch.load(f"{model_prefix}_fold_{i}_best.pth")) 
            m.eval()
            models_list.append(m.to(DEVICE))
        except FileNotFoundError:
            print(f"Warning: Model file not found for {model_prefix} fold {i}. Skipping.")
            continue
            
    if not models_list:
        raise ValueError(f"No models found for {model_prefix}. Did you train them first?")

    all_logits = []
    
    with torch.no_grad():
        for imgs, names in tqdm(loader, desc=f"Predicting {model_prefix} (TTA/Ens)"):
            imgs = imgs.to(DEVICE)
            
            # --- TTA: 4 Views (Original, FlipH, FlipV, Rot90) ---
            imgs_lr = torch.flip(imgs, [3])
            imgs_ud = torch.flip(imgs, [2])
            imgs_rot = torch.rot90(imgs, 1, [2, 3])
            
            inputs = [imgs, imgs_lr, imgs_ud, imgs_rot]
            
            # Use 1 unit output (Logits, not probabilities)
            batch_logits = torch.zeros((imgs.size(0), 1)).to(DEVICE)
            
            # Ensemble over 5 Folds + TTA over 4 views
            for m in models_list:
                for inp in inputs:
                    # m(inp) returns logits
                    batch_logits += m(inp)
            
            # Average: Total = (N_FOLDS * N_TTA)
            batch_logits /= (len(models_list) * len(inputs))
            
            all_logits.append(batch_logits.squeeze(1).cpu()) # Store raw logits for majority voting
            
    return torch.cat(all_logits, dim=0)

In [ ]:
# --- 7. SUBMISSION (Hierarchical Prediction & TTA/Ensemble) ---
def submit():
    print("--- ⚙️ Generating Submission with Hierarchical TTA/Ensemble ---")
    
    # --- A. DATA SETUP ---

    df_test_processed = get_processed_test_data()
    
    if df_test_processed.empty:
        raise ValueError("Test preprocessing returned an empty DataFrame. Cannot generate submission.")
        
    
    # 2. Setup DataLoader using the processed DataFrame and the correct path
    test_ds = TissueDataset(
        dataframe=df_test_processed,                              # FIX: Use 'dataframe' instead of 'df'
        precomputed_dir=TEST_PRECOMPUTED_DIR,                     # FIX: Use 'precomputed_dir' instead of 'img_dir'
        transform=get_transforms()[1],                            # Validation/Test transforms
        is_test=True, 
        target_label_column='sample_index'                        # Target column is irrelevant for test
    )
    
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    # --- B. HIERARCHICAL PREDICTION (C1 -> C2/C3) ---
    
    # 1. Classify C1 (Triage: Luminal Group vs. Aggressive Group)
    # The prefix is the key from CLASSIFIER_CONFIGS
    logits_c1 = predict_with_ensemble_tta(loader, 'C1_TRIAGE')
    
    # C1 prediction (0: Luminal Group, 1: Aggressive Group)
    # We use sigmoid and a threshold (0.5) to get the prediction.
    preds_c1 = (torch.sigmoid(logits_c1) >= 0.5).long() 
    
    
    # 2. Classify C2 (Luminal Subtypes) and C3 (Aggressive Subtypes)
    
    # Initialize the final result array (will hold the final 4-class prediction)
    final_4class_preds = torch.zeros_like(preds_c1, device=DEVICE) # Initialize on device
    
    # --- C2: Predict on Luminal Group (where C1_pred == 0) ---
    luminal_indices = (preds_c1 == 0).nonzero(as_tuple=True)[0]
    
    if len(luminal_indices) > 0:
        # FIX 1: Use df_test_processed instead of undefined test_df
        luminal_df = df_test_processed.iloc[luminal_indices].reset_index(drop=True)
        # FIX 2: C2 prediction must read from TEST_PRECOMPUTED_DIR, not PRECOMPUTED_DIR (which is for train)
        luminal_ds = TissueDataset(
            dataframe=luminal_df, 
            precomputed_dir=TEST_PRECOMPUTED_DIR, 
            transform=get_transforms()[1], 
            is_test=True
        )
        luminal_loader = DataLoader(luminal_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        
        logits_c2 = predict_with_ensemble_tta(luminal_loader, 'C2_LUMINAL_SUBTYPE')
        # C2 prediction (0: Luminal A, 1: Luminal B)
        # Final 4-class mapping: 0 -> Luminal A, 1 -> Luminal B
        
        # We need the predictions in the original test_df order (luminal_indices)
        preds_c2 = (torch.sigmoid(logits_c2) >= 0.5).long() # 0 or 1
        
        # Mapping: Luminal A (0) is original class 0, Luminal B (1) is original class 1
        # The final index must be placed back into the final_4class_preds array
        final_4class_preds[luminal_indices] = preds_c2.to(DEVICE) # 0 or 1
        
    
    # --- C3: Predict on Aggressive Group (where C1_pred == 1) ---
    aggressive_indices = (preds_c1 == 1).nonzero(as_tuple=True)[0]
    
    if len(aggressive_indices) > 0:
            # FIX 3: Use df_test_processed instead of undefined test_df
            aggressive_df = df_test_processed.iloc[aggressive_indices].reset_index(drop=True)
            # FIX 4: C3 prediction must read from TEST_PRECOMPUTED_DIR, not PRECOMPUTED_DIR (which is for train)
            aggressive_ds = TissueDataset(
                dataframe=aggressive_df, 
                precomputed_dir=TEST_PRECOMPUTED_DIR, 
                transform=get_transforms()[1], 
                is_test=True
            )
            aggressive_loader = DataLoader(aggressive_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            
            logits_c3 = predict_with_ensemble_tta(aggressive_loader, 'C3_AGGRESSIVE_SUBTYPE')
            preds_c3 = (torch.sigmoid(logits_c3) >= 0.5).long() # 0 or 1
            
            # Shift binary prediction by +2: 0->2, 1->3
            final_4class_preds[aggressive_indices] = preds_c3.to(DEVICE) + 2
        
    
    # --- C. FINAL OUTPUT MAPPING ---
    
    # Define the final 4 class names (ordered by the index 0, 1, 2, 3)
    FINAL_4CLASS_NAMES = [
        'Luminal A',        # C2_pred == 0
        'Luminal B',        # C2_pred == 1
        'HER2(+)',          # C3_pred == 0 (index 2)
        'Triple negative'   # C3_pred == 1 (index 3)
    ]
    
    # Map the final numerical prediction (0, 1, 2, or 3) back to class strings
    final_labels_text = [FINAL_4CLASS_NAMES[i.item()] for i in final_4class_preds]
    
    # Create Submission File
    submission_df = pd.DataFrame({
        'sample_index': df_test_processed['sample_index'], # Filenames from test_df
        'label': final_labels_text
    })
    
    submission_df.to_csv('submission.csv', index=False)
    print("Done. Submission file 'submission.csv' created.")
    return submission_df

In [ ]:
submit()